In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

sns.set_style('darkgrid')

In [3]:
df = pd.read_csv("IMDB Dataset.csv", encoding="utf-8", on_bad_lines="skip")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [5]:
df.sample(5)

,review,sentiment
29605,This is one of those games where you love it t...,positive
37279,"Is it a remake og the Thing (1982/1951), i thi...",negative
39399,But the fun is in the journey.<br /><br />I fo...,positive
23584,"Luckily, not knowing anything about this movie...",positive
10597,"Man, this is a hard DVD to come by. I could on...",positive


In [6]:
df.sentiment.value_counts()

,count
sentiment,
positive,25000
negative,25000


In [8]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg])


In [9]:
train,test = train_test_split(
    df_reviews,
    test_size=0.33,
    random_state=42
)

train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [10]:
train_y.value_counts()

,count
sentiment,
negative,3378
positive,3322


In [11]:
tfidf = TfidfVectorizer(stop_words='english')

train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [12]:
train_x_vector.shape

(6700, 44107)

In [13]:
svc = SVC(kernel='linear')

svc.fit(train_x_vector, train_y)

SVC(kernel='linear')

In [14]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all'])))

['positive']
['positive']
['negative']


In [15]:
svc.score(test_x_vector, test_y)

0.8706060606060606

In [16]:
f1_score(
    test_y,
    svc.predict(test_x_vector),
    labels=['positive','negative'],
    average=None
)

array([0.87400413, 0.86701962])

In [17]:
print(
    classification_report(
        test_y,
        svc.predict(test_x_vector),
        labels=['positive','negative']
    )
)

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



In [18]:
conf_mat = confusion_matrix(
    test_y,
    svc.predict(test_x_vector),
    labels=['positive','negative']
)

conf_mat

array([[1481,  197],
       [ 230, 1392]])